In [0]:
# caminho Edu
bureau_full = spark.read.parquet("/Volumes/workspace/hackathon_2025/default/source/base_score_bureau_movel/part-00000-b6bf6bdd-9a2d-4bc5-a97c-fa8553906bc9-c000.snappy.parquet")

In [0]:
# caminho Michael
bureau_full = spark.read.parquet("/Volumes/hackathon_2025/default/source/base_score_bureau_movel_full/")

In [0]:
%sql
SHOW CATALOGS

In [0]:
bureau_full.show(15)

In [0]:
display(bureau_full)

In [0]:
bureau_full.createOrReplaceTempView("bureau_full")

#Select Distinct

In [0]:
%sql
SELECT DISTINCT FLAG_INSTALACAO FROM bureau_full

In [0]:
%sql
SELECT DISTINCT FPD FROM bureau_full

In [0]:
%sql
SELECT DISTINCT PROD FROM bureau_full

In [0]:
%sql
SELECT DISTINCT flag_mig2 FROM bureau_full

#Checks

In [0]:
%sql
SELECT Safra,  
      COUNT(*)
        
FROM bureau_full
GROUP BY 1
ORDER BY 1

In [0]:
%sql
SELECT 
      FPD,  
      COUNT(*)
        
FROM bureau_full
GROUP BY ALL
ORDER BY ALL

In [0]:
%sql
SELECT Safra,
      FPD,  
      COUNT(*)
        
FROM bureau_full
WHERE FPD IS NOT NULL
GROUP BY ALL
ORDER BY ALL

In [0]:
%sql
SELECT Safra,
      ROUND(SUM(FPD)/COUNT(*),4) AS pct_FPD,  
      COUNT(*)
        
FROM bureau_full
WHERE FPD IS NOT NULL
GROUP BY ALL
ORDER BY ALL

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Fazer visão %Default por faixa de score
SELECT 
    MIN(SCORE_01),
    MAX(SCORE_01),
    AVG(SCORE_01),
    MIN(SCORE_02),
    MAX(SCORE_02),
    AVG(SCORE_02)
FROM bureau_full

# 20260108 - Michael

In [0]:
%sql
-- verificação das colunas de score, quantidade de nulos/vazios e zeros
SELECT
  COUNT(*) AS total_linhas,

  SUM(CASE WHEN SCORE_01 IS NULL OR TRIM(SCORE_01) = '' THEN 1 ELSE 0 END) AS score01_null_ou_vazio,
  SUM(CASE WHEN CAST(SCORE_01 AS INT) = 0 THEN 1 ELSE 0 END) AS score01_igual_zero,

  SUM(CASE WHEN SCORE_02 IS NULL OR TRIM(SCORE_02) = '' THEN 1 ELSE 0 END) AS score02_null_ou_vazio
FROM bureau_full;

In [0]:
%sql
-- verificando o percentil das colunas SCORE_01 e SCORE_02
SELECT
  percentile_approx(CAST(SCORE_01 AS INT), array(0.01,0.05,0.25,0.50,0.75,0.95,0.99), 10000) AS pctl_score01,
  percentile_approx(CAST(SCORE_02 AS INT), array(0.01,0.05,0.25,0.50,0.75,0.95,0.99), 10000) AS pctl_score02
FROM bureau_full;

In [0]:
%sql
-- quantidade de zeros para FDP=0 e FDP=1
SELECT
  FPD,
  COUNT(*) AS qtd
FROM bureau_full
WHERE CAST(SCORE_01 AS INT) = 0
GROUP BY FPD
ORDER BY FPD;

In [0]:
%sql
-- Comparar total de linhas vs chaves únicas
SELECT
  COUNT(*) AS total_linhas,
  COUNT(DISTINCT CONCAT(NUM_CPF, '#', SAFRA)) AS chaves_unicas_cpf_safra,
  COUNT(*) - COUNT(DISTINCT CONCAT(NUM_CPF, '#', SAFRA)) AS linhas_duplicadas
FROM bureau_full;

In [0]:
%sql
SELECT
  COUNT(*) AS total,
  COUNT(DISTINCT CONCAT(NUM_CPF,'#',SAFRA)) AS distinct_cpf_safra
FROM bureau_full;

In [0]:
%sql
SELECT
  FLAG_INSTALACAO,
  SUM(CASE WHEN FPD IS NULL OR TRIM(FPD)='' THEN 1 ELSE 0 END) AS fpd_null,
  SUM(CASE WHEN FPD IS NOT NULL AND TRIM(FPD)<>'' THEN 1 ELSE 0 END) AS fpd_not_null,
  COUNT(*) AS total
FROM bureau_full
GROUP BY FLAG_INSTALACAO
ORDER BY FLAG_INSTALACAO;